##**Timeout wrapper using asyncio.wait_for(coroutine, timeout=2.0)**

In [1]:
import asyncio

async def slow_vlm_network_call():
    """
    Simulates a heavy Vision-Language Model request that hangs due to
    poor cloud server response times.
    """
    try:
        print("[VLM Network] Sending camera frames to cloud brain...")
        await asyncio.sleep(5.0)  # Simulates a heavy, bad 5-second delay
        print("[VLM Network] Success! Received movement coordinates.")
        return "MOVE_FORWARD"
    except asyncio.CancelledError:
        print("[VLM Network] TIMEOUT ENFORCED! Cloud task cancelled mid-flight.")
        raise

async def main_robot_controller():
    print("--- INITIATING MISSION CONTROLLER ---")
    try:
        # Wrap the 5.0s task in a strict 2.0s deadline guard
        print("[Controller] Firing VLM request with a strict 2.0s deadline...")
        decision = await asyncio.wait_for(slow_vlm_network_call(), timeout=2.0)
        print(f"[Controller] Robot proceeding with command: {decision}")

    except asyncio.TimeoutError:
        print("\n[Controller] ALERT: VLM failed to respond within 2.0 seconds!")
        print("[Controller] FALLBACK RUNNING: Activating local obstacle detection sensors...")
        print("[Controller] FALLBACK RUNNING: Executing safe physical stop.")

# Run the timeout test
await main_robot_controller()

--- INITIATING MISSION CONTROLLER ---
[Controller] Firing VLM request with a strict 2.0s deadline...
[VLM Network] Sending camera frames to cloud brain...
[VLM Network] TIMEOUT ENFORCED! Cloud task cancelled mid-flight.

[Controller] ALERT: VLM failed to respond within 2.0 seconds!
[Controller] FALLBACK RUNNING: Activating local obstacle detection sensors...
[Controller] FALLBACK RUNNING: Executing safe physical stop.
